# Label Dataset Audit

This notebook validates the canonical feature dataset against the feature schema, and audits the current labeling progress.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve().parent))
from src.data_loader import load_v2_features
from src.feature_schema import FeatureSchemaValidator
from src.label_validation import LabelValidator

FEATURES_PATH = "../data/processed/features/event_features_v2.parquet"
LABELS_PATH = "data/labels_template.csv"
FEATURE_SCHEMA = "configs/feature_schema.json"
LABEL_SCHEMA = "data/label_schema.json"


## 1. Load Datasets

In [ ]:
print("Loading canonical feature dataset...")
try:
    features_df = load_v2_features(FEATURES_PATH)
    print(f"Feature Dataset Dimensions: {features_df.shape}")
except Exception as e:
    print(f"Error loading features: {e}")
    features_df = pd.DataFrame()

print("Loading labels...")
try:
    labels_df = pd.read_csv(LABELS_PATH)
    print(f"Labels Dataset Dimensions: {labels_df.shape}")
except Exception as e:
    print(f"Error loading labels: {e}")
    labels_df = pd.DataFrame()


## 2. Feature Schema Validation

In [ ]:
if not features_df.empty:
    f_validator = FeatureSchemaValidator(FEATURE_SCHEMA)
    # We want to test that if we try to use ALL columns, it correctly rejects the bad ones.
    try:
        f_validator.validate_features(list(features_df.columns))
        print("All features valid (Unexpected, as leakage features should be in the dataset).")
    except ValueError as e:
        print("Feature schema validation correctly caught excluded features:")
        print(e)


## 3. Label Validation & Coverage

In [ ]:
if not features_df.empty and not labels_df.empty:
    l_validator = LabelValidator(LABEL_SCHEMA)
    errors = l_validator.validate_labels(labels_df, features_df)
    if errors:
        print("Label Validation Errors:")
        for err in errors:
            print(f" - {err}")
    else:
        print("Labels are fully valid against the schema.")
        
    # Labeled events stats
    num_labeled = len(labels_df)
    num_total = len(features_df)
    print(f"\nLabeled events: {num_labeled}")
    print(f"Unlabeled events: {num_total - num_labeled}")
    print(f"Label Coverage: {(num_labeled / num_total) * 100:.4f}%")
    
    if num_labeled > 0 and 'final_label' in labels_df.columns:
        print("\nClass Distribution:")
        print(labels_df['final_label'].value_counts(dropna=False))
elif labels_df.empty:
    print("\nLabeled events: 0")
    if not features_df.empty:
        print(f"Unlabeled events: {len(features_df)}")
        print("Label Coverage: 0.0%")
